# 🧠 Convolutional Neural Networks
**Course:** Deep Learning  
**Lecture:** 3 - Convolutional Neural Networks (CNNs)  
**Part:** a - The Convolution Operation

---

> **Where we are:** We know how dense layers work, but they ignore spatial structure; now we introduce convolution to process grid-like data like images efficiently.

**What you'll learn:**
- 2D convolution from scratch (no Keras): sliding kernel, element-wise multiply + sum
- Key parameters and their effect on output size:
  - `padding='valid'`: output shrinks - formula: $(H - K + 1) \times (W - K + 1)$
  - `padding='same'`: output same size - how padding is added
  - `stride=1 vs 2 vs 3`: formula: $\lfloor(H - K)/S + 1\rfloor$
- Filters as feature detectors: edge detection kernels (Sobel, Laplacian)
- Multiple filters = multiple channels: how the depth dimension grows
- Receptive field: what area of the input does a neuron 'see'

**Exam relevance:** ⭐⭐⭐ (Extremely relevant. You must be able to calculate output shapes and parameter counts!)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import mnist

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

## 📖 1. The 2D Convolution Operation
A convolution operation slides a small matrix (the **kernel** or **filter**) over an input image. At each location, we perform an element-wise multiplication between the kernel and the image patch, and sum the results.

Given an input $I$ and kernel $K$ of size $m \times n$:
$ S(i, j) = (I * K)(i, j) = \sum_m \sum_n I(i + m, j + n) K(m, n) $

**Receptive Field:** The area of the input volume that a particular feature affects. For a $3 \times 3$ kernel, a single output neuron "sees" a $3 \times 3$ patch of the input.

## 🔀 Your Options: Convolution Implementation
At this step, you can choose between:
| Option | Pros | Cons | When to use |
|---|---|---|---|
| From Scratch (Loops) | Deepens understanding | Very slow, CPU only | Learning the mechanics |
| `tf.nn.conv2d` | Highly optimized, runs on GPU | Hides complexity | Real-world applications |

In [ ]:
# ✏️ Your Turn: 2D Convolution from scratch

# We will use a real MNIST digit
(x_train, _), (_, _) = mnist.load_data()
image = x_train[0] / 255.0 # (28, 28)

def conv2d_scratch(image, kernel, stride=1, padding='valid'):
    # ### YOUR CODE HERE ###
    # 1. Handle padding ('same' means add zero-padding so output shape matches formula)
    # 2. Calculate output shape: H_out = (H_in - K + 2*P) // S + 1
    # 3. Create nested loops to slide the kernel over the image
    # 4. Multiply and sum
    pass

# Tests below should pass once implemented
# print(conv2d_scratch(image, np.ones((3,3))).shape)

In [ ]:
# ✅ Solution
def conv2d_scratch(image, kernel, stride=1, padding='valid'):
    # Get dimensions
    h_in, w_in = image.shape
    k_h, k_w = kernel.shape
    
    # Add padding if requested
    if padding == 'same':
        pad_h = (k_h - 1) // 2
        pad_w = (k_w - 1) // 2
        img_padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant')
    else:
        img_padded = image
        
    h_padded, w_padded = img_padded.shape
    
    # Calculate output dimensions
    h_out = (h_padded - k_h) // stride + 1
    w_out = (w_padded - k_w) // stride + 1
    
    output = np.zeros((h_out, w_out))
    
    # Slide the kernel
    for i in range(h_out):
        for j in range(w_out):
            patch = img_padded[i*stride : i*stride+k_h, j*stride : j*stride+k_w]
            output[i, j] = np.sum(patch * kernel)
            
    return output

# Define edge detection kernels
sobel_x = np.array([[-1, 0, 1], 
                    [-2, 0, 2], 
                    [-1, 0, 1]])

sobel_y = np.array([[-1, -2, -1], 
                    [ 0,  0,  0], 
                    [ 1,  2,  1]])


In [ ]:
# 🔬 Comparison Cell: Original vs Sobel-X vs Sobel-Y
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image, cmap='gray')
axes[0].set_title(f"Original {image.shape}")

out_x_valid = conv2d_scratch(image, sobel_x, padding='valid')
axes[1].imshow(out_x_valid, cmap='gray')
axes[1].set_title(f"Sobel-X (Vertical Edges)\nValid Pad: {out_x_valid.shape}")

out_y_same = conv2d_scratch(image, sobel_y, padding='same', stride=1)
axes[2].imshow(out_y_same, cmap='gray')
axes[2].set_title(f"Sobel-Y (Horizontal Edges)\nSame Pad: {out_y_same.shape}")

plt.show()

# Table comparing output shapes vs padding/stride
import pandas as pd
shapes = {
    'Padding': ['valid', 'same', 'valid'],
    'Stride': [1, 1, 2],
    'Output Size': [
        conv2d_scratch(image, sobel_x, padding='valid', stride=1).shape,
        conv2d_scratch(image, sobel_y, padding='same', stride=1).shape,
        conv2d_scratch(image, sobel_x, padding='valid', stride=2).shape
    ]
}
display(pd.DataFrame(shapes))

# Verify against TF
img_tf = tf.reshape(tf.constant(image, dtype=tf.float32), (1, 28, 28, 1))
sobel_x_tf = tf.reshape(tf.constant(sobel_x, dtype=tf.float32), (3, 3, 1, 1))

out_tf = tf.nn.conv2d(img_tf, sobel_x_tf, strides=[1, 1, 1, 1], padding='VALID')
out_tf = tf.squeeze(out_tf).numpy()

np.testing.assert_allclose(out_x_valid, out_tf, rtol=1e-5)
print("✅ Manual convolution matches tf.nn.conv2d perfectly!")

## 🎓 Exam Corner

### Question 1 (Practical)
Input: (28, 28, 1). Kernel: (3, 3). Padding='valid', stride=1. What is the output shape?

<details><summary>💡 Answer</summary>

Output shape formula: $H_{out} = \lfloor(H_{in} - K)/S + 1\rfloor$
$H_{out} = (28 - 3)/1 + 1 = 26$
So, the output shape is (26, 26, 1).

</details>

### Question 2 (Practical)
Input: (64, 64, 3). Kernel: (5, 5), 32 filters. Padding='same', stride=2. Output shape?

<details><summary>💡 Answer</summary>

Padding 'same' with stride 2 halves the spatial dimensions: $64 / 2 = 32$.
The number of channels becomes the number of filters = 32.
Output shape: (32, 32, 32).

</details>

### Question 3 (Conceptual)
How many trainable parameters does a Conv2D layer have, given kernel_size=(3,3), in_channels=16, out_channels=32?

<details><summary>💡 Answer</summary>

Params per filter = $3 \times 3 \times 16 = 144$ weights + 1 bias = 145.
Total params = $145 \times 32 = 4,640$.

</details>